# NVDA-CMX Stakeholder Dashboard

This notebook includes:

- Dark-mode multi-select stakeholder comparison chart
- Prediction summary and per-ticker prediction charts
- Read-only audit panel during normal widget changes
- **Log Current Predictions** button to append snapshots intentionally
- CSV-backed prediction log for internal validation over time

## Packages
Install these if needed:

```bash
pip install yfinance pandas plotly ipywidgets scikit-learn
```

Run the code cell below, then use the selector and button in the dashboard.


In [3]:

import math
import os
from datetime import datetime
from zoneinfo import ZoneInfo

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

PREDICTION_LOG_PATH = "prediction_log.csv"
ALL_TICKERS = ["DELL", "HPE", "IBM", "NTNX", "PWR", "SMCI", "^SPX", "^IXIC", "AMD", "NTAP", "HTHIY", "VG" ]

display(HTML("""
<style>
    body,
    .jp-Notebook,
    .jp-Cell,
    .jp-OutputArea,
    .jp-OutputArea-output,
    .jp-OutputArea-child,
    .jupyter-widgets,
    .jupyter-widgets-output-area,
    .jp-RenderedHTMLCommon,
    .cell-output,
    .output,
    .output_area {
        background-color: #1b2430 !important;
        color: #e5e7eb !important;
    }

    .widget-label {
        color: #e5e7eb !important;
        font-weight: 600 !important;
    }

    select {
        background-color: #111827 !important;
        color: #e5e7eb !important;
        border: 1px solid #3b4758 !important;
        border-radius: 8px !important;
        padding: 6px !important;
    }

    select:focus {
        outline: none !important;
        border-color: #60a5fa !important;
        box-shadow: 0 0 0 1px #60a5fa !important;
    }

    table.dataframe td,
    table.dataframe th {
        color: #f8fafc !important;
    }
</style>
"""))

def load_price_data(tickers, period="6mo"):
    data = yf.download(tickers, period=period, auto_adjust=True, progress=False)["Close"]
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])
    return data.dropna(how="all")

price_data = load_price_data(ALL_TICKERS, period="6mo")
normalized = price_data.div(price_data.iloc[0]).sub(1)

header = widgets.HTML(
    value="""
    <div style="
        font-family: Inter, Arial, sans-serif;
        font-size: 20px;
        font-weight: 700;
        color: #f8fafc;
        margin: 0 0 14px 0;
        padding: 14px 18px;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
        border: 1px solid #3b4758;
        border-radius: 12px;
        width: 100%;
        box-sizing: border-box;
        box-shadow: 0 8px 24px rgba(0,0,0,0.35);
        letter-spacing: 0.2px;
    ">
        NVDA-CMX Stakeholder Dashboard
    </div>
    """
)

instruction = widgets.HTML(
    value="""
    <div style="
        font-family: Inter, Arial, sans-serif;
        font-size: 13px;
        color: #d1d5db;
        margin: 0 0 12px 0;
        padding: 10px 14px;
        background: linear-gradient(135deg, #111827 0%, #1f2937 100%);
        border: 1px solid #3b4758;
        border-radius: 10px;
        width: fit-content;
        box-shadow: 0 4px 14px rgba(0,0,0,0.35);
        letter-spacing: 0.2px;
    ">
        <span style="font-weight:600; color:#f8fafc;">Selection Controls:</span>
        Hold <span style="color:#93c5fd; font-weight:600;">Shift</span> to multi-select.
        Hold <span style="color:#93c5fd; font-weight:600;">Ctrl</span> to compare in the dropdown.
    </div>
    """
)

selector_label = widgets.HTML(
    value="""
    <div style="
        font-family: Inter, Arial, sans-serif;
        font-size: 12px;
        font-weight: 600;
        color: #cbd5e1;
        margin: 0 0 8px 2px;
        text-transform: uppercase;
        letter-spacing: 0.8px;
    ">
        Ticker Universe
    </div>
    """
)

ticker_selector = widgets.SelectMultiple(
    options=ALL_TICKERS,
    value=tuple(ALL_TICKERS[:3]),
    description="",
    rows=len(ALL_TICKERS),
    layout=widgets.Layout(width="320px", height="220px")
)

log_button = widgets.Button(
    description="Log Current Predictions",
    tooltip="Append the current selection's predictions to the audit log",
    button_style="",
    layout=widgets.Layout(width="220px", height="38px")
)
log_button.style.button_color = "#2563eb"

status_banner = widgets.HTML(
    value="""
    <div style="
        font-family: Inter, Arial, sans-serif;
        font-size: 12px;
        color: #cbd5e1;
        margin: 10px 0 0 0;
    ">
        Audit panel is read-only during normal widget changes. Use the button to log a new prediction snapshot.
    </div>
    """
)

button_panel = widgets.VBox(
    [log_button, status_banner],
    layout=widgets.Layout(
        padding="0 0 0 14px",
        justify_content="center"
    )
)

control_row = widgets.HBox(
    [
        widgets.VBox(
            [selector_label, ticker_selector],
            layout=widgets.Layout(
                padding="14px 14px 10px 14px",
                border="1px solid #3b4758",
                border_radius="12px",
                background_color="#1b2430",
                width="360px"
            )
        ),
        button_panel
    ],
    layout=widgets.Layout(align_items="flex-start")
)

comparison_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #3b4758",
        padding="12px",
        margin="12px 0 12px 0",
        background_color="#1b2430"
    )
)

table_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #3b4758",
        padding="12px",
        margin="0 0 12px 0",
        background_color="#1b2430"
    )
)

prediction_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #3b4758",
        padding="12px",
        margin="0 0 12px 0",
        background_color="#1b2430"
    )
)

audit_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #3b4758",
        padding="12px",
        background_color="#1b2430"
    )
)

message_output = widgets.Output(
    layout=widgets.Layout(margin="0 0 10px 0")
)

def build_comparison_figure(selected):
    fig = go.Figure()

    for ticker in selected:
        if ticker in normalized.columns:
            fig.add_trace(
                go.Scatter(
                    x=normalized.index,
                    y=normalized[ticker] * 100,
                    mode="lines",
                    name=ticker,
                    line=dict(width=2.5)
                )
            )

    fig.update_layout(
        title=dict(text="NVDA-CMX Stakeholder Performance", font=dict(size=22, color="#f8fafc")),
        xaxis_title="Date (6month)",
        yaxis_title="Net Change %",
        template="plotly_dark",
        hovermode="x unified",
        height=650,
        paper_bgcolor="#0b1220",
        plot_bgcolor="#111827",
        font=dict(color="#e5e7eb"),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            bgcolor="rgba(0,0,0,0)",
            font=dict(color="#e5e7eb")
        ),
        margin=dict(l=60, r=30, t=90, b=60)
    )

    fig.update_yaxes(
        ticksuffix="%",
        showgrid=True,
        gridcolor="rgba(148,163,184,0.15)",
        zeroline=True,
        zerolinecolor="rgba(148,163,184,0.25)"
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(148,163,184,0.10)")
    return fig

def run_prediction_model(ticker):
    df = yf.download(ticker, period="6mo", auto_adjust=True, progress=False)

    if df.empty:
        return None

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    required_cols = [c for c in ["Close", "Volume"] if c in df.columns]
    if set(required_cols) != {"Close", "Volume"}:
        return None

    df = df[["Close", "Volume"]].dropna().copy()
    df["return_1d"] = df["Close"].pct_change(1)
    df["return_5d"] = df["Close"].pct_change(5)
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_10"] = df["Close"].rolling(10).mean()
    df["ma_ratio_5_10"] = df["ma_5"] / df["ma_10"] - 1
    df["volatility_5"] = df["return_1d"].rolling(5).std()
    df["volume_change"] = df["Volume"].pct_change()
    df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
    df = df.dropna().copy()

    features = ["return_1d", "return_5d", "ma_ratio_5_10", "volatility_5", "volume_change"]
    if len(df) < 40:
        return None

    split_idx = int(len(df) * 0.8)
    train = df.iloc[:split_idx].copy()
    test = df.iloc[split_idx:].copy()

    if train.empty or test.empty:
        return None

    X_train = train[features]
    y_train = train["target"]
    X_test = test[features]
    y_test = test["target"]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=5,
        random_state=42
    )

    model.fit(X_train, y_train)
    test["pred"] = model.predict(X_test)
    test["prob_up"] = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, test["pred"])
    latest_features = df[features].iloc[[-1]]
    latest_prob_up = model.predict_proba(latest_features)[0, 1]
    latest_prediction = "UP" if latest_prob_up >= 0.5 else "DOWN"

    return {
        "ticker": ticker,
        "test_frame": test,
        "accuracy": accuracy,
        "latest_prob_up": latest_prob_up,
        "latest_prediction": latest_prediction
    }

def build_prediction_figure(results):
    n = len(results)
    cols = 2
    rows = math.ceil(n / cols)

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"{r['ticker']} Prediction View" for r in results],
        shared_xaxes=False,
        vertical_spacing=0.14
    )

    for i, result in enumerate(results):
        row = i // cols + 1
        col = i % cols + 1
        test = result["test_frame"]
        buy_signals = test[test["pred"] == 1]
        sell_signals = test[test["pred"] == 0]

        fig.add_trace(
            go.Scatter(
                x=test.index,
                y=test["Close"],
                mode="lines",
                name=f"{result['ticker']} Close",
                line=dict(width=2.2),
                showlegend=False
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Scatter(
                x=buy_signals.index,
                y=buy_signals["Close"],
                mode="markers",
                name=f"{result['ticker']} Pred Up",
                marker=dict(symbol="triangle-up", size=10),
                showlegend=False
            ),
            row=row,
            col=col
        )

        fig.add_trace(
            go.Scatter(
                x=sell_signals.index,
                y=sell_signals["Close"],
                mode="markers",
                name=f"{result['ticker']} Pred Down",
                marker=dict(symbol="triangle-down", size=10),
                showlegend=False
            ),
            row=row,
            col=col
        )

    fig.update_layout(
        title=dict(text="Selected Ticker Prediction Models", font=dict(size=20, color="#f8fafc")),
        template="plotly_dark",
        hovermode="x unified",
        height=max(500, rows * 350),
        paper_bgcolor="#0b1220",
        plot_bgcolor="#111827",
        font=dict(color="#e5e7eb"),
        margin=dict(l=60, r=30, t=80, b=50)
    )

    fig.update_xaxes(title_text="Date", showgrid=True, gridcolor="rgba(148,163,184,0.10)")
    fig.update_yaxes(
        title_text="Adjusted Close",
        showgrid=True,
        gridcolor="rgba(148,163,184,0.15)",
        zeroline=True,
        zerolinecolor="rgba(148,163,184,0.25)"
    )

    for annotation in fig.layout.annotations:
        annotation.font = dict(size=13, color="#cbd5e1")

    return fig

def build_summary_table(results):
    rows = [{
        "Ticker": r["ticker"],
        "Test Accuracy": f"{r['accuracy']:.2%}",
        "Latest Up Probability": f"{r['latest_prob_up']:.2%}",
        "Next Session Forecast Prediction": r["latest_prediction"]
    } for r in results]
    return pd.DataFrame(rows).sort_values("Latest Up Probability", ascending=False).reset_index(drop=True)

def render_dark_table(df):
    styled = (
        df.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "table", "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Inter, Arial, sans-serif"),
                ("background-color", "#0f172a"),
                ("color", "#f8fafc"),
                ("border", "1px solid #3b4758"),
                ("border-radius", "10px"),
                ("overflow", "hidden")
            ]},
            {"selector": "thead th", "props": [
                ("background-color", "#1e293b"),
                ("color", "#f8fafc"),
                ("font-weight", "700"),
                ("padding", "10px 12px"),
                ("border-bottom", "1px solid #3b4758"),
                ("text-align", "left")
            ]},
            {"selector": "tbody td", "props": [
                ("padding", "10px 12px"),
                ("border-bottom", "1px solid #1f2937"),
                ("text-align", "left"),
                ("color", "#f8fafc"),
                ("background-color", "#0f172a")
            ]},
            {"selector": "tbody tr:nth-child(even) td", "props": [
                ("background-color", "#162030"),
                ("color", "#f8fafc")
            ]},
            {"selector": "tbody tr:nth-child(odd) td", "props": [
                ("background-color", "#0f172a"),
                ("color", "#f8fafc")
            ]},
            {"selector": "tbody tr:hover td", "props": [
                ("background-color", "#1f2937"),
                ("color", "#ffffff")
            ]}
        ])
    )
    display(styled)

def generate_prediction_snapshot(tickers):
    rows = []
    run_timestamp = datetime.now(ZoneInfo("America/New_York")).strftime("%Y-%m-%d %H:%M:%S %Z")

    for ticker in tickers:
        result = run_prediction_model(ticker)
        if result is None:
            continue

        latest_signal_date = result["test_frame"].index.max()

        rows.append({
            "run_timestamp_est": run_timestamp,
            "ticker": ticker,
            "signal_date": pd.Timestamp(latest_signal_date).strftime("%Y-%m-%d"),
            "predicted_next_session_direction": result["latest_prediction"],
            "predicted_up_probability": round(result["latest_prob_up"], 6),
            "test_accuracy_at_prediction_time": round(result["accuracy"], 6)
        })

    return pd.DataFrame(rows)

def append_prediction_log(snapshot_df, log_path=PREDICTION_LOG_PATH):
    if snapshot_df.empty:
        return pd.DataFrame()

    if os.path.exists(log_path):
        existing = pd.read_csv(log_path)
        combined = pd.concat([existing, snapshot_df], ignore_index=True)
        combined = combined.drop_duplicates(
            subset=["run_timestamp_est", "ticker", "signal_date"],
            keep="last"
        )
    else:
        combined = snapshot_df.copy()

    combined.to_csv(log_path, index=False)
    return combined

def validate_prediction_log(log_path=PREDICTION_LOG_PATH):
    if not os.path.exists(log_path):
        return pd.DataFrame()

    log_df = pd.read_csv(log_path)
    if log_df.empty:
        return pd.DataFrame()

    validated_rows = []

    for ticker in log_df["ticker"].unique():
        ticker_log = log_df[log_df["ticker"] == ticker].copy()
        price_df = yf.download(ticker, period="1y", auto_adjust=True, progress=False)

        if price_df.empty:
            continue

        if isinstance(price_df.columns, pd.MultiIndex):
            price_df.columns = price_df.columns.get_level_values(0)

        price_df = price_df[["Close"]].dropna().copy()
        price_df.index = pd.to_datetime(price_df.index)

        for _, row in ticker_log.iterrows():
            signal_date = pd.to_datetime(row["signal_date"])
            future_dates = price_df.index[price_df.index > signal_date]
            signal_close = price_df.loc[signal_date, "Close"] if signal_date in price_df.index else None

            if len(future_dates) == 0 or signal_close is None:
                next_session_date = None
                next_session_close = None
                actual_direction = None
                prediction_correct = None
            else:
                next_session_date = future_dates[0]
                next_session_close = price_df.loc[next_session_date, "Close"]
                actual_direction = "UP" if next_session_close > signal_close else "DOWN"
                prediction_correct = actual_direction == row["predicted_next_session_direction"]

            validated_rows.append({
                **row.to_dict(),
                "signal_close": signal_close,
                "next_session_date": pd.Timestamp(next_session_date).strftime("%Y-%m-%d") if next_session_date is not None else None,
                "next_session_close": next_session_close,
                "actual_next_session_direction": actual_direction,
                "prediction_correct": prediction_correct
            })

    validated_df = pd.DataFrame(validated_rows)
    validated_df.to_csv(log_path, index=False)
    return validated_df

def build_prediction_audit_summary(validated_df):
    if validated_df.empty:
        return pd.DataFrame()

    completed = validated_df.dropna(subset=["prediction_correct"]).copy()
    if completed.empty:
        return pd.DataFrame()

    completed["prediction_correct"] = completed["prediction_correct"].astype(bool)

    summary = (
        completed.groupby("ticker")
        .agg(
            total_predictions=("prediction_correct", "count"),
            correct_predictions=("prediction_correct", "sum"),
            avg_predicted_up_probability=("predicted_up_probability", "mean"),
            avg_test_accuracy_at_prediction_time=("test_accuracy_at_prediction_time", "mean")
        )
        .reset_index()
    )

    summary["realized_accuracy"] = summary["correct_predictions"] / summary["total_predictions"]
    return summary.sort_values("realized_accuracy", ascending=False).reset_index(drop=True)

def show_banner(output_widget, text, fg="#fde68a", bg="#1f1a12", border="#92400e"):
    output_widget.clear_output(wait=True)
    with output_widget:
        display(HTML(f"""
            <div style="
                font-family: Inter, Arial, sans-serif;
                color: {fg};
                background: {bg};
                border: 1px solid {border};
                border-radius: 10px;
                padding: 12px 14px;
            ">
                {text}
            </div>
        """))

def render_audit_panel():
    audit_output.clear_output(wait=True)

    with audit_output:
        display(HTML("""
            <div style="
                font-family: Inter, Arial, sans-serif;
                font-size: 14px;
                font-weight: 700;
                color: #f8fafc;
                margin: 0 0 12px 0;
                letter-spacing: 0.3px;
            ">
                Model Audit Panel
            </div>
        """))

        validated_df = validate_prediction_log()

        if validated_df.empty:
            display(HTML("""
                <div style="
                    font-family: Inter, Arial, sans-serif;
                    color: #cbd5e1;
                    background: #0f172a;
                    border: 1px solid #3b4758;
                    border-radius: 10px;
                    padding: 12px 14px;
                ">
                    No audit history available yet. Use <b>Log Current Predictions</b> to create the first snapshot.
                </div>
            """))
            return

        completed = validated_df.dropna(subset=["prediction_correct"]).copy()
        total_logged = len(validated_df)
        total_completed = len(completed)
        realized_hit_rate = completed["prediction_correct"].astype(bool).mean() if total_completed > 0 else None

        metric_html = f"""
        <div style="display:flex; gap:14px; flex-wrap:wrap; margin-bottom:16px;">
            <div style="background:#0f172a; border:1px solid #3b4758; border-radius:10px; padding:12px 16px; min-width:180px;">
                <div style="color:#94a3b8; font-size:12px; font-family:Inter, Arial, sans-serif;">Total Logged Predictions</div>
                <div style="color:#f8fafc; font-size:22px; font-weight:700; font-family:Inter, Arial, sans-serif;">{total_logged}</div>
            </div>
            <div style="background:#0f172a; border:1px solid #3b4758; border-radius:10px; padding:12px 16px; min-width:180px;">
                <div style="color:#94a3b8; font-size:12px; font-family:Inter, Arial, sans-serif;">Validated Predictions</div>
                <div style="color:#f8fafc; font-size:22px; font-weight:700; font-family:Inter, Arial, sans-serif;">{total_completed}</div>
            </div>
            <div style="background:#0f172a; border:1px solid #3b4758; border-radius:10px; padding:12px 16px; min-width:180px;">
                <div style="color:#94a3b8; font-size:12px; font-family:Inter, Arial, sans-serif;">Realized Hit Rate</div>
                <div style="color:#f8fafc; font-size:22px; font-weight:700; font-family:Inter, Arial, sans-serif;">{f"{realized_hit_rate:.2%}" if realized_hit_rate is not None else "N/A"}</div>
            </div>
        </div>
        """
        display(HTML(metric_html))

        display(HTML("""
            <div style="
                font-family: Inter, Arial, sans-serif;
                font-size: 13px;
                font-weight: 700;
                color: #f8fafc;
                margin: 8px 0 10px 0;
            ">
                Per-Ticker Accuracy Leaderboard
            </div>
        """))

        summary_df = build_prediction_audit_summary(validated_df)
        if summary_df.empty:
            display(HTML("""
                <div style="
                    font-family: Inter, Arial, sans-serif;
                    color: #cbd5e1;
                    margin-bottom: 12px;
                ">
                    Not enough validated predictions yet for ticker-level accuracy.
                </div>
            """))
        else:
            leaderboard_df = summary_df.copy()
            leaderboard_df["realized_accuracy"] = leaderboard_df["realized_accuracy"].map(lambda x: f"{x:.2%}")
            leaderboard_df["avg_predicted_up_probability"] = leaderboard_df["avg_predicted_up_probability"].map(lambda x: f"{x:.2%}")
            leaderboard_df["avg_test_accuracy_at_prediction_time"] = leaderboard_df["avg_test_accuracy_at_prediction_time"].map(lambda x: f"{x:.2%}")
            render_dark_table(leaderboard_df)

        display(HTML("""
            <div style="
                font-family: Inter, Arial, sans-serif;
                font-size: 13px;
                font-weight: 700;
                color: #f8fafc;
                margin: 16px 0 10px 0;
            ">
                Last 10 Predictions
            </div>
        """))

        last_10 = validated_df.sort_values("run_timestamp_est", ascending=False).head(10).copy()
        display_cols = [
            "run_timestamp_est",
            "ticker",
            "signal_date",
            "predicted_next_session_direction",
            "predicted_up_probability",
            "actual_next_session_direction",
            "prediction_correct"
        ]

        for col in display_cols:
            if col not in last_10.columns:
                last_10[col] = None

        last_10 = last_10[display_cols].copy()
        last_10["predicted_up_probability"] = last_10["predicted_up_probability"].map(
            lambda x: f"{x:.2%}" if pd.notnull(x) else None
        )
        render_dark_table(last_10)

def refresh_dashboard():
    selected = list(ticker_selector.value)

    comparison_output.clear_output(wait=True)
    table_output.clear_output(wait=True)
    prediction_output.clear_output(wait=True)
    message_output.clear_output(wait=True)

    if not selected:
        show_banner(comparison_output, "Select at least one ticker.", fg="#fca5a5", bg="#1f1720", border="#7f1d1d")
        return

    with comparison_output:
        build_comparison_figure(selected).show()

    results = []
    for ticker in selected:
        result = run_prediction_model(ticker)
        if result is not None:
            results.append(result)

    if not results:
        show_banner(table_output, "No prediction results available for the current selection.")
        return

    with table_output:
        display(HTML("""
            <div style="
                font-family: Inter, Arial, sans-serif;
                font-size: 14px;
                font-weight: 700;
                color: #f8fafc;
                margin: 0 0 10px 0;
                letter-spacing: 0.3px;
            ">
                Prediction Summary
            </div>
        """))
        render_dark_table(build_summary_table(results))

    with prediction_output:
        build_prediction_figure(results).show()

    render_audit_panel()

def handle_selection_change(change):
    refresh_dashboard()

def handle_log_button(_):
    selected = list(ticker_selector.value)
    message_output.clear_output(wait=True)

    if not selected:
        with message_output:
            display(HTML("""
                <div style="
                    font-family: Inter, Arial, sans-serif;
                    color: #fca5a5;
                    background: #1f1720;
                    border: 1px solid #7f1d1d;
                    border-radius: 10px;
                    padding: 12px 14px;
                    margin: 0 0 10px 0;
                ">
                    Select at least one ticker before logging a snapshot.
                </div>
            """))
        return

    snapshot_df = generate_prediction_snapshot(selected)
    if snapshot_df.empty:
        with message_output:
            display(HTML("""
                <div style="
                    font-family: Inter, Arial, sans-serif;
                    color: #fde68a;
                    background: #1f1a12;
                    border: 1px solid #92400e;
                    border-radius: 10px;
                    padding: 12px 14px;
                    margin: 0 0 10px 0;
                ">
                    No snapshot was logged because no valid predictions were available.
                </div>
            """))
        return

    append_prediction_log(snapshot_df)
    render_audit_panel()

    tickers_text = ", ".join(snapshot_df["ticker"].tolist())
    run_time = snapshot_df["run_timestamp_est"].iloc[0]

    with message_output:
        display(HTML(f"""
            <div style="
                font-family: Inter, Arial, sans-serif;
                color: #bbf7d0;
                background: #0f1f17;
                border: 1px solid #166534;
                border-radius: 10px;
                padding: 12px 14px;
                margin: 0 0 10px 0;
            ">
                Logged {len(snapshot_df)} prediction snapshot(s) at <b>{run_time}</b> for: <b>{tickers_text}</b>
            </div>
        """))

ticker_selector.observe(handle_selection_change, names="value")
log_button.on_click(handle_log_button)

dashboard = widgets.VBox(
    [
        header,
        instruction,
        control_row,
        message_output,
        comparison_output,
        table_output,
        prediction_output,
        audit_output
    ],
    layout=widgets.Layout(
        width="100%",
        padding="10px",
        margin="0",
        border="1px solid #3b4758",
        border_radius="14px",
        background_color="#1b2430"
    )
)

display(dashboard)
refresh_dashboard()
